# Scam Title Classification with PyIBL and Sentence Transformers

This notebook shows how to use embedding-based similarity inside a PyIBL agent to classify short email titles as **scam** or **safe**.

You will:
- load a tiny labeled dataset
- configure embedding-aware similarity on the `title` attribute
- train online and plot a learning curve
- visualize how the sentence-transformer fits into the IBL decision loop

## Requirements

Install optional dependencies if needed:

- `pip install sentence-transformers`
- `pip install matplotlib`

In [ ]:
from pathlib import Path
import csv
import importlib
import random

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise SystemExit(
        "This notebook requires matplotlib. Install it with: pip install matplotlib"
    ) from exc

try:
    SentenceTransformer = importlib.import_module("sentence_transformers").SentenceTransformer
except ImportError as exc:
    raise SystemExit(
        "This notebook requires sentence-transformers. Install it with: pip install sentence-transformers"
    ) from exc

from pyibl import Agent

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_SEED = 7

In [ ]:
data_path = Path("scam_titles.csv")
if not data_path.exists():
    data_path = Path("examples/embedding/scam_titles.csv")

with open(data_path, newline="", encoding="utf-8") as infile:
    examples = list(csv.DictReader(infile))

print(f"Loaded {len(examples)} labeled titles from {data_path}")
examples[:3]

## Build the Embedding-Aware Agent

The agent stores experiences over `(title, label)` choices.

Embedding-based similarity is enabled on `title` so semantically similar titles can partially match previously seen examples.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(model_name)

def make_agent():
    # default_utility avoids cold-start errors before memory is populated.
    agent = Agent(attributes=["title", "label"], mismatch_penalty=1, default_utility=0.5)
    agent.embedding(function=encoder)
    agent.embedding("title")
    return agent

agent = make_agent()
print("Embedding-enabled attributes:", agent.embedding())

## Train Online and Track Learning

Each trial asks the agent to choose between two labels (`scam` vs `safe`) for one title.

The response reward is `1` for correct and `0` for incorrect, then we plot accuracy over time.

In [ ]:
def run_epoch(agent, rows, rng):
    shuffled = rows[:]
    rng.shuffle(shuffled)
    correct = 0
    trial_hits = []

    for row in shuffled:
        options = [
            {"title": row["title"], "label": "scam"},
            {"title": row["title"], "label": "safe"},
        ]
        prediction = agent.choose(options)
        is_correct = prediction["label"] == row["label"]
        agent.respond(1 if is_correct else 0)

        correct += int(is_correct)
        trial_hits.append(int(is_correct))

    return correct / len(shuffled), trial_hits

def train_agent(rows, epochs=25, seed=RANDOM_SEED):
    rng = random.Random(seed)
    trained = make_agent()
    epoch_accuracy = []
    trial_accuracy = []

    for _ in range(epochs):
        acc, hits = run_epoch(trained, rows, rng)
        epoch_accuracy.append(acc)
        trial_accuracy.extend(hits)

    return trained, epoch_accuracy, trial_accuracy

trained_agent, epoch_accuracy, trial_accuracy = train_agent(examples, epochs=25)
cumulative_accuracy = np.cumsum(trial_accuracy) / np.arange(1, len(trial_accuracy) + 1)

print(f"Epoch 1 accuracy:  {epoch_accuracy[0]:.2f}")
print(f"Epoch {len(epoch_accuracy)} accuracy: {epoch_accuracy[-1]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(range(1, len(epoch_accuracy) + 1), epoch_accuracy, marker="o", linewidth=2, color="#0B5FA5")
axes[0].axhline(0.5, color="#888888", linestyle="--", linewidth=1, label="chance baseline")
axes[0].set_title("Accuracy by Training Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc="lower right")

axes[1].plot(range(1, len(cumulative_accuracy) + 1), cumulative_accuracy, linewidth=2.5, color="#2E8B57")
axes[1].set_title("Cumulative Accuracy Across Trials")
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Cumulative accuracy")
axes[1].set_ylim(0, 1.05)

fig.suptitle("Learning Curve for Scam vs Safe Title Classification", fontsize=13)
fig.tight_layout()
plt.show()

## How SentenceTransformer Fits into the IBL Loop

This diagram summarizes the full decision process used in the notebook.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

def draw_box(ax, x, y, w, h, text, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.07",
                           linewidth=1.2, edgecolor="#1A1A1A", facecolor=color)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=10)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>",
                                mutation_scale=12, linewidth=1.5, color="#333333"))

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.set_xlim(0, 15)
ax.set_ylim(0, 6)
ax.axis("off")

draw_box(ax, 0.5, 3.8, 2.6, 1.2, "Email title text", "#F7E4BC")
draw_box(ax, 3.7, 3.8, 3.3, 1.2, "SentenceTransformer\nembedding vector", "#D7E8FF")
draw_box(ax, 7.6, 3.8, 3.2, 1.2, "IBL memory\n(title, label, utility)", "#E1F4D8")
draw_box(ax, 11.3, 3.8, 3.0, 1.2, "Choose label\nscam or safe", "#FADADD")

draw_box(ax, 7.6, 1.0, 3.2, 1.2, "Similarity + blended\nvalue computation", "#F3E5F5")
draw_box(ax, 11.3, 1.0, 3.0, 1.2, "Feedback reward\nrespond(1 or 0)", "#FFF3B5")

draw_arrow(ax, 3.1, 4.4, 3.7, 4.4)
draw_arrow(ax, 7.0, 4.4, 7.6, 4.4)
draw_arrow(ax, 10.8, 4.4, 11.3, 4.4)
draw_arrow(ax, 9.2, 3.8, 9.2, 2.2)
draw_arrow(ax, 10.8, 1.6, 11.3, 1.6)
draw_arrow(ax, 12.8, 1.0, 9.2, 1.0)
draw_arrow(ax, 9.2, 2.2, 9.2, 3.8)

ax.text(0.5, 0.25,
        "Embedding similarity allows partial matching on semantically related titles,\n"
        "which helps the IBL agent generalize beyond exact string matches.",
        fontsize=10, color="#222222")

ax.set_title("SentenceTransformer Inside the IBL Decision Cycle", fontsize=14, pad=10)
plt.show()

## Quick Prediction Demo

Try a new title with the trained agent.

In [ ]:
query = "Urgent: your mailbox storage is full, verify now"
options = [
    {"title": query, "label": "scam"},
    {"title": query, "label": "safe"},
]

choice, details = trained_agent.choose(options, details=True)
print(f"Predicted label for query: {choice['label']}")
print("Top retrieval summary:")
for item in details:
    print(f"  {item['choice']['label']}: blended_value={item['blended_value']:.3f}")

# Clear pending decision without external label feedback.
trained_agent.respond()